# أثر × ALLaM — Hail heritage test

Does **ALLaM-7B** (SDAIA / HUMAIN, Apache-2.0) know Hail's heritage? This asks it 24 questions built from
Athr's own verified site facts (`src/data/sites.js`), scores the answers **twice** — ALLaM alone, then ALLaM given Athr's verified site text (grounded) — and asks 3 open "tour guide" questions.

**Kaggle:** verify your phone first (profile → Settings → Phone verification), then ⚙️ *Settings* → **Accelerator: GPU T4 x2** (or any GPU) · **Internet: On** → *Run All*.
**Colab:** *Runtime → Change runtime type → T4 GPU* → *Run all*. On a single 15–16 GB GPU the model loads in 8-bit automatically.
First run downloads ~15 GB (≈5–10 min). Results land in `/kaggle/working/` (Kaggle *Output* panel) or the Colab *Files* panel. Full run ≈ 15–20 min.

In [ ]:
!pip -q install -U "transformers>=4.40.1" accelerate bitsandbytes

In [ ]:
import torch, json, re, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU (Kaggle) / Runtime -> Change runtime type -> T4 (Colab)"
MODEL = "humain-ai/ALLaM-7B-Instruct-preview"
gpu_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())) / 1e9
tok = AutoTokenizer.from_pretrained(MODEL)
if gpu_gb >= 20:   # Kaggle T4 x2 (~30 GB): full fp16 quality
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
else:              # one 15-16 GB GPU (Kaggle single T4 / P100, Colab T4): 8-bit fits with room to generate
    model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto",
                                                 quantization_config=BitsAndBytesConfig(load_in_8bit=True))
model.eval()
print(f"GPU memory {gpu_gb:.0f} GB -> {'fp16' if gpu_gb >= 20 else '8-bit'}; loaded on", {str(p.device) for p in model.parameters()})

In [ ]:
def ask(question, max_new_tokens=200):
    msgs = [{"role": "user", "content": question}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt", return_token_type_ids=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)  # greedy = repeatable
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask("كيف أجهز كوب شاهي؟"))  # the model card's own smoke test

In [ ]:
EVAL = {
 "sites": [
  {
   "id": "jubbah",
   "name": "نقوش جبة الصخرية",
   "context": "الموقع: نقوش جبة الصخرية — جبل أم سنمان · موقع تراث عالمي (يونسكو)\nالمكان: جبة، شمال غرب حائل\nالحقبة: ~10,000 سنة مضت (العصر الحجري الحديث)\nالإدراج في اليونسكو: 2015 م\nعدد اللوحات التقديري: أكثر من 5,400 نقش\nأقدم النقوش: نحو 10,000 سنة\nالمسافة من حائل: 95 كم شمال غربًا\nحين كانت الصحراء بحيرة: قبل نحو عشرة آلاف عام لم تكن جبةُ رملًا كما نراها اليوم، بل بحيرةً عذبة تحفّ بها مروجٌ خضراء تقصدها قطعان البقر الوحشيّ والمها والنعام؛ وعلى ضفاف تلك البحيرة استقرّ إنسان ما قبل التاريخ، فرفع رأسه إلى واجهات جبل أُمّ سِنمان الصخرية فوجدها لوحًا ينتظر يده.\nأُمّ سِنمان: الجبل الذي يتكلّم: يمتدّ جبل أُمّ سِنمان شمال قرية جبة كسورٍ صخريّ ضخم تكسو واجهاتِه آلافُ النقوش المحفورة بالطَّرْق والحزّ، فترى فيها رجالًا بأجسادٍ فارعة وأكتافٍ عريضة، ونساءً بضفائر طويلة، وحيواناتٍ نُقشت بدقّةٍ تشريحية مذهلة من بقرٍ وحشيّ بقرونه المقوّسة ووعولٍ وأسودٍ ونعام؛ حتى إنّ أكبر هذه الأشكال يتجاوز طوله المترين، فقد نُقش ليُرى من بعيد كأنّه إعلانٌ عن هويّة.\nمن الصورة إلى الحرف: وما يجعل جبةَ استثنائيةً أنّها لا توثّق زمنًا واحدًا، وإنّما تسجّل انتقال الإنسان من الرسم إلى الكتابة؛ إذ تعلو النقوشَ الحيوانيةَ القديمة كتاباتٌ ثمودية ونبطية وعربية مبكّرة حفرها مسافرون مرّوا بعدها بآلاف السنين، حتى صار الجدار الواحد هنا دفترَ زوّارٍ عمره عشرة آلاف عام.\nاعترافٌ عالميّ: وفي عام 2015م أدرجت اليونسكو «الفنّ الصخريّ في منطقة حائل» ضمن قائمة التراث العالميّ، وشمل الإدراجُ جبلَ أُمّ سِنمان في جبة وجبلَي المنجور وراط في الشويمس، فكان رابع موقعٍ سعوديّ يدخل القائمة، وأوّل اعترافٍ دوليّ بأنّ صحراء النفود كانت يومًا مسرحًا لحضارةٍ كاملة."
  },
  {
   "id": "aarif",
   "name": "قلعة أَعَيْرِف",
   "context": "الموقع: قلعة أَعَيْرِف — حصن التلّة · أكثر من قرنين\nالمكان: وسط مدينة حائل\nالحقبة: أكثر من 200 عام\nالعمر التقديري: أكثر من 200 عام\nالارتفاع عن المدينة: نحو 30 مترًا\nالصعود: درجٌ فقط · لا طريق للسيارة\nمادة البناء: طين وتبن وجذوع أثل\nعين حائل التي لا تنام: على قمّة تلّةٍ صخرية تشرف على مدينة حائل تقف قلعة أَعَيْرِف، وقد بناها أهلُها قبل أكثر من قرنين لتكون عينًا ساهرة على الطرق المؤدّية إليها؛ فمن أبراجها كان الحرّاس يرصدون القوافل القادمة عبر درب زبيدة، ويوقدون النار ليالي رمضان إيذانًا بالإفطار، وهي عادةٌ ما زالت حيّةً إلى اليوم في المدفع الذي يُطلق من التلّة نفسها.\nالطين الذي صمد قرنين: بُنيت القلعة من الطين المخلوط بالتبن، بجدرانٍ يتجاوز سُمكها المتر وتدعمها جذوع الأثل، وليست هذه التقنية بدائيةً كما تبدو؛ فالجدار الطينيّ السميك يحفظ برودة الليل نهارًا وحرارة النهار ليلًا، فيصنع بذلك مكيّفًا طبيعيًا في قلب صحراءَ تتجاوز حرارتها الخمسين درجة صيفًا.\nسبعون درجة إلى أفضل إطلالة: ولا يُصعد إليها إلا على قدمين، فالدرج المنحوت في التلّة يبلغ نحو سبعين درجة تتسلّقها في دقائق؛ وما إن تبلغ السطح حتى تنبسط حائل تحتك كاملةً، فترى القشلة غربًا مستطيلًا طينيًا واضح الحدود، وجبال أجا خلفها تحجب الأفق، والمدينة الحديثة تمتدّ بينهما.\nالتلّة التي اختارت نفسها: ولم يكن اختيار هذه التلّة ذوقًا معماريًّا، بل هي المرتفع الطبيعيّ الوحيد وسط سهل حائل؛ وثلاثون مترًا فوق السهل تكفي لتغطية الأفق في كل اتجاه، وقاعدتها صخرٌ صلد لا يُنقب من تحته ولا يُنصب عليه سلّمٌ ولا برجُ حصار. ثم دارت الأيام فامتدّت المدينة حول التلّة وتجاوزتها، حتى صارت القلعة التي بُنيت لتراقب الصحراء تطلّ اليوم على شوارع مزدحمة — والمكان نفسه، والوظيفة انقلبت."
  },
  {
   "id": "qishlah",
   "name": "قصر القشلة",
   "context": "الموقع: قصر القشلة — 142 غرفة · 1360هـ / 1941م\nالمكان: وسط مدينة حائل\nالحقبة: 1360هـ / 1941م\nعدد الغرف: 142 غرفة\nأبراج المراقبة: 8 أبراج\nسنة البناء: 1360هـ / 1941م\nمادة البناء: طين وتبن وجذوع أثل\nمدينةٌ داخل جدار: القشلة قلعةٌ مستطيلة ضخمة أمر ببنائها الملك عبدالعزيز عام 1360هـ لتكون ثكنةً عسكرية، تضمّ 142 غرفة موزّعة على طابقين، وثمانية أبراج للمراقبة في زواياها وأضلاعها، ومسجدًا في الداخل، وفناءً واسعًا تتقاطع فيه الأروقة؛ حتى إنّ المشي في ممرّاتها الطينية الطويلة يشبه التجوّل في مدينةٍ مصغّرة لا في مبنًى واحد.\nالطين حين يُبنى بمسطرة: وما يميّز القشلة عن العمارة الطينية القديمة أنّها بُنيت بتخطيطٍ هندسيّ صارم لا بالتوسّع التدريجيّ، فالأروقة متوازية والغرف متطابقة المقاسات والأبراج موزّعة بانتظام؛ وهي بذلك تقف على الحدّ الفاصل بين عمارة نجد التقليدية والعمارة الحديثة المخطّطة، مبنيّةً بالطين والتبن وجذوع الأثل نفسها التي بُنيت بها قلعة أَعَيْرِف قبلها بقرنين.\nثمانية أبراج بلا نقطةٍ عمياء: واللافت في أبراجها الثمانية أنّ كلّ نافذةٍ فيها صُمّمت بزاويةٍ مختلفة عن جارتها، لتغطّي مجتمعةً كلّ اتّجاهٍ حول القلعة دون أن تترك نقطةً عمياء واحدة؛ وهو حسابٌ بصريّ دقيق أُنجز دون أدوات مسحٍ حديثة، ويمكنك التحقّق منه بنفسك إن وقفت في الفناء وتتبّعت خطوط الرؤية من برجٍ إلى برج.\nمن ثكنةٍ إلى ذاكرة: ولمّا انتهى دورها العسكريّ تحوّلت القشلة إلى واحدٍ من أهمّ معالم حائل السياحية، تُقام في فنائها الفعاليات الثقافية والأسواق التراثية وفعاليات مهرجان حائل؛ فصار المبنى الذي بُني ليُغلق دون الناس مكانًا لا معنى له إلا بهم."
  },
  {
   "id": "aja",
   "name": "جبال أجا",
   "context": "الموقع: جبال أجا — سلسلة الجرانيت الأحمر · غرب حائل\nالمكان: غرب مدينة حائل\nالحقبة: تكوين جيولوجي عمره أكثر من 500 مليون سنة\nأعلى قمة: نحو 1,500 م فوق سطح البحر\nنوع الصخر: جرانيت وردي وبركانيات قديمة\nالامتداد: أكثر من 100 كم\nالارتباط التاريخي: قبيلة طيء · حاتم الطائي\nأجا وسلمى: جبلان وقصّة حبّ: تروي كتب الأخبار العربية أنّ «أجا» رجلٌ أحبّ امرأةً تُدعى «سلمى»، وأنّ قومهما فرّقوا بينهما فهربا، فأُدركا وقُتلا ومُسخ كلٌّ منهما جبلًا؛ فصار أجا غربيّ حائل وسلمى شرقيّها، يفصل بينهما سهلٌ واسع لا يلتقيان عبره أبدًا. وهي أسطورةٌ في أصلها، غير أنّها ترسّخت حتى غدا الاسمان عَلَمين جغرافيّين رسميّين في كلّ خرائط الجزيرة.\nحصن طيّءٍ الطبيعيّ: ولم تكن أجا جبلًا جميلًا فحسب، بل كانت قلعة قبيلة طيّء التي نزلت المنطقة في القرون الأولى قبل الإسلام، فجعلت من شعابها الضيّقة ومسالكها الوعرة حصنًا يستحيل اقتحامه؛ ومن هذه الجبال خرج حاتمٌ الطائيّ الذي صار كرمه مثلًا يُضرب في العربية إلى اليوم، وخرج ابنه عديٌّ الذي وفد على النبيّ ﷺ. أمّا تسمية «جبل شمّر» التي اشتهرت لاحقًا فتشمل أجا وسلمى معًا.\nجرانيتٌ بلون النار: صخور أجا جرانيتٌ ورديّ إلى أحمر داكن، تكوّن قبل أكثر من خمسمئة مليون سنة من صهارةٍ بردت ببطءٍ في أعماق الأرض قبل أن ترفعها الحركات الأرضية، ولهذا جاءت بلورات الكوارتز والفلسبار فيه كبيرةً واضحةً للعين المجرّدة؛ وعند الغروب تعكس هذه الصخور الضوء فتبدو السلسلة كلّها كأنّها تشتعل، وتلك هي اللحظة التي ينتظرها المصوّرون.\nماءٌ في قلب الحجر: وبين شعاب أجا مساراتٌ وأوديةٌ وغُدرانٌ تحتفظ بالماء بعد الأمطار، وحولها نخيلٌ وقرًى صغيرة عاشت على هذه المياه قرونًا؛ والمشي في هذه الشعاب صباحًا تجربةٌ تختلف تمامًا عن صورة الصحراء النمطية، فثمّة ظلالٌ باردة، وصدًى للصوت بين الجدران الصخرية، ونقوشٌ قديمة تفاجئك على صخرةٍ في منتصف الطريق."
  },
  {
   "id": "museum",
   "name": "متحف حائل الإقليمي",
   "context": "الموقع: متحف حائل الإقليمي — خلاصة المنطقة تحت سقف واحد\nالمكان: مدينة حائل\nالحقبة: من عصور ما قبل التاريخ إلى العصر الحديث\nأقسام العرض: ما قبل التاريخ · الممالك · الإسلامي · التراث الشعبي\nمدة الزيارة المثالية: 90 دقيقة\nمناسب للأطفال: نعم — قاعات مكيّفة ومسار قصير\nالدخول: مجاني\nلماذا تبدأ من هنا: أكثر ما يقع فيه زوّار حائل من خطأٍ أنّهم يقصدون جبةَ أوّلًا، والمتحف الإقليميّ هو الذي يمنحك المفتاح؛ فترى فيه نماذج النقوش مشروحة، وخرائطَ توضّح كيف كانت البحيرات تغطّي النفود، وقطعًا أصلية من مواقع المنطقة. وبعد ساعةٍ هنا تصير كلّ صخرةٍ تراها في الخارج مفهومةً بدل أن تكون رسمًا غامضًا.\nمن الصوّان إلى الحديد: تتدرّج قاعات المتحف زمنيًا، فتبدأ بأدواتٍ صوّانية من العصر الحجريّ تخبرك كيف صنع الإنسان الأوّل سكّينه، ثمّ تنتقل إلى الفخار والأدوات البرونزية، فالنقوش الثمودية والنبطية المنقولة أو المنسوخة، وصولًا إلى القسم الإسلاميّ حيث تظهر حائل محطّةً رئيسية على درب زبيدة، وهو طريق الحجّ القادم من الكوفة إلى مكّة.\nدرب زبيدة يمرّ من هنا: ومن أهمّ ما يعرضه المتحف قصّةُ درب زبيدة، ذلك الطريق الذي أمرت زبيدةُ زوجُ هارون الرشيد بتجهيزه للحجّاج ببرك الماء والآبار والمنائر التي تهدي المسافرين ليلًا، ومحطّات استراحةٍ على مسافاتٍ محسوبة؛ ومرور هذا الطريق بمنطقة حائل هو ما جعلها مركزًا تجاريًا وثقافيًا قرونًا طويلة، لا محطّةً عابرة.\nالتراث الحيّ: أمّا القسم الأخير فمخصّصٌ للتراث الشعبيّ من أدوات القهوة والدلال، والأزياء المطرّزة، وأدوات الزراعة والريّ القديمة، ومقتنيات البيت الحائليّ؛ وهنا يتّضح أنّ كرم حاتمٍ الطائيّ لم يكن أسطورةً أدبية فحسب، بل ثقافةَ ضيافةٍ كاملة لها أدواتها وطقوسها التي ما زالت تُمارَس في بيوت حائل اليوم."
  },
  {
   "id": "shuwaymis",
   "name": "نقوش الشويمس",
   "context": "الموقع: نقوش الشويمس — جبل المنجور وراط · موقع تراث عالمي (يونسكو)\nالمكان: الشويمس، جنوب غرب حائل\nالحقبة: ~10,000 سنة مضت (العصر الحجري الحديث)\nالإدراج في اليونسكو: 2015 م (مع جبة)\nالمكوّنان: جبل المنجور · جبل راط\nالمسافة من حائل: 250 كم جنوب غربًا\nالوصول: مرشد معتمد · جزء من الطريق ترابي\nالنصف المنسيّ من الإدراج: حين أدرجت اليونسكو «الفنّ الصخريّ في منطقة حائل» عام 2015م لم يكن الإدراج لجبةَ وحدها، بل لموقعين اثنين: جبل أُمّ سِنمان في جبة، وجبلَي المنجور وراط في الشويمس؛ غير أنّ الشويمس ظلّت الأقلّ زيارةً لبعدها ووعورة طريقها، فبقيت نقوشُها أسلمَ حالًا وأقلَّ عبثًا — وهي اليوم من أفضل ما حُفظ من فنّ العصر الحجريّ الحديث في الجزيرة كلّها.\nأودية تُقرأ كصفحات: ولا تقع نقوش الشويمس على واجهةٍ واحدة كما في جبة، وإنّما تتوزّع على جدران أوديةٍ ضيّقة عميقة تمشي بينها فتنكشف لك لوحةٌ بعد لوحة؛ فترى قطعان البقر الوحشيّ والأسود والفهود والنعام، وأشكالًا بشريةً تصطاد بالقوس، ومشاهدَ كاملة لا رموزًا متفرّقة — حتى إنّ بعض الألواح تروي حدثًا واحدًا من أوّله إلى آخره.\nالكلب الذي غيّر التاريخ: وفي الشويمس تحديدًا عُثر على ما يُعدّ من أقدم التصويرات المعروفة للكلاب المدجَّنة في العالم: كلابٌ ذات آذانٍ منتصبة وذيولٍ ملتفّة تحيط بصيّادٍ يشدّ قوسه، وبعضها مربوطٌ إليه بخطوطٍ فُسّرت بأنّها مقاود؛ ولو صحّ التفسير فإنّ هذه الجدران توثّق لحظةً من أهمّ لحظات علاقة الإنسان بالحيوان قبل أكثر من ثمانية آلاف عام.\nلماذا هنا: والسبب في هذا الازدحام على جدارٍ بعينه أنّ الشويمس كانت ملتقى مياه: أوديةٌ تنحدر من الحرّات البركانية فتجتمع في مسايل تحتفظ بالماء بعد المطر، فتقصدها الحيوانات فيقصدها الصيّادون؛ فما النقوش إلا سجلٌّ لمن وقف هنا ينتظر — والجدار الذي ترسم عليه هو الجدار الذي تستظلّ به."
  }
 ],
 "questions": [
  {
   "site_id": "jubbah",
   "site": "نقوش جبة الصخرية",
   "expected": "2015 م",
   "q": "ما الإدراج في اليونسكو لموقع «نقوش جبة الصخرية» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "jubbah",
   "site": "نقوش جبة الصخرية",
   "expected": "أكثر من 5,400 نقش",
   "q": "ما عدد اللوحات التقديري لموقع «نقوش جبة الصخرية» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "jubbah",
   "site": "نقوش جبة الصخرية",
   "expected": "نحو 10,000 سنة",
   "q": "ما أقدم النقوش لموقع «نقوش جبة الصخرية» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "jubbah",
   "site": "نقوش جبة الصخرية",
   "expected": "95 كم شمال غربًا",
   "q": "ما المسافة من حائل لموقع «نقوش جبة الصخرية» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "aarif",
   "site": "قلعة أَعَيْرِف",
   "expected": "أكثر من 200 عام",
   "q": "ما العمر التقديري لموقع «قلعة أَعَيْرِف» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "aarif",
   "site": "قلعة أَعَيْرِف",
   "expected": "نحو 30 مترًا",
   "q": "ما الارتفاع عن المدينة لموقع «قلعة أَعَيْرِف» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "aarif",
   "site": "قلعة أَعَيْرِف",
   "expected": "درجٌ فقط · لا طريق للسيارة",
   "q": "ما الصعود لموقع «قلعة أَعَيْرِف» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "aarif",
   "site": "قلعة أَعَيْرِف",
   "expected": "طين وتبن وجذوع أثل",
   "q": "ما مادة البناء لموقع «قلعة أَعَيْرِف» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "qishlah",
   "site": "قصر القشلة",
   "expected": "142 غرفة",
   "q": "ما عدد الغرف لموقع «قصر القشلة» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "qishlah",
   "site": "قصر القشلة",
   "expected": "8 أبراج",
   "q": "ما أبراج المراقبة لموقع «قصر القشلة» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "qishlah",
   "site": "قصر القشلة",
   "expected": "1360هـ / 1941م",
   "q": "ما سنة البناء لموقع «قصر القشلة» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "qishlah",
   "site": "قصر القشلة",
   "expected": "طين وتبن وجذوع أثل",
   "q": "ما مادة البناء لموقع «قصر القشلة» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "aja",
   "site": "جبال أجا",
   "expected": "نحو 1,500 م فوق سطح البحر",
   "q": "ما أعلى قمة لموقع «جبال أجا» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "aja",
   "site": "جبال أجا",
   "expected": "جرانيت وردي وبركانيات قديمة",
   "q": "ما نوع الصخر لموقع «جبال أجا» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "aja",
   "site": "جبال أجا",
   "expected": "أكثر من 100 كم",
   "q": "ما الامتداد لموقع «جبال أجا» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "aja",
   "site": "جبال أجا",
   "expected": "قبيلة طيء · حاتم الطائي",
   "q": "ما الارتباط التاريخي لموقع «جبال أجا» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "museum",
   "site": "متحف حائل الإقليمي",
   "expected": "ما قبل التاريخ · الممالك · الإسلامي · التراث الشعبي",
   "q": "ما أقسام العرض لموقع «متحف حائل الإقليمي» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "museum",
   "site": "متحف حائل الإقليمي",
   "expected": "90 دقيقة",
   "q": "ما مدة الزيارة المثالية لموقع «متحف حائل الإقليمي» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "museum",
   "site": "متحف حائل الإقليمي",
   "expected": "نعم — قاعات مكيّفة ومسار قصير",
   "q": "ما مناسب للأطفال لموقع «متحف حائل الإقليمي» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "museum",
   "site": "متحف حائل الإقليمي",
   "expected": "مجاني",
   "q": "ما الدخول لموقع «متحف حائل الإقليمي» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "shuwaymis",
   "site": "نقوش الشويمس",
   "expected": "2015 م (مع جبة)",
   "q": "ما الإدراج في اليونسكو لموقع «نقوش الشويمس» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "shuwaymis",
   "site": "نقوش الشويمس",
   "expected": "جبل المنجور · جبل راط",
   "q": "ما المكوّنان لموقع «نقوش الشويمس» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "shuwaymis",
   "site": "نقوش الشويمس",
   "expected": "250 كم جنوب غربًا",
   "q": "ما المسافة من حائل لموقع «نقوش الشويمس» في منطقة حائل؟ أجب باختصار."
  },
  {
   "site_id": "shuwaymis",
   "site": "نقوش الشويمس",
   "expected": "مرشد معتمد · جزء من الطريق ترابي",
   "q": "ما الوصول لموقع «نقوش الشويمس» في منطقة حائل؟ أجب باختصار."
  }
 ]
}

In [ ]:
AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def numbers(s):
    s = s.translate(AR_DIGITS).replace(",", "").replace("٬", "")
    s = re.sub(r"(\d+(?:\.\d+)?)\s*(آلاف|ألف|الف)", lambda m: str(int(float(m.group(1)) * 1000)), s)  # "10 آلاف" = 10000
    return re.findall(r"\d+(?:\.\d+)?", s)

def score(expected, answer):
    """✅ every number in the expected fact is in the answer · ❌ one is missing · 👀 no numbers: judge by eye"""
    exp = numbers(expected)
    if not exp:
        return "👀"
    got = set(numbers(answer))
    return "✅" if all(n in got for n in exp) else "❌"

CONTEXT = {s["id"]: s["context"] for s in EVAL["sites"]}

def run(make_prompt, label):
    rows = []
    for i, item in enumerate(EVAL["questions"], 1):
        t = time.time()
        a = ask(make_prompt(item))
        rows.append({**item, "answer": a, "score": score(item["expected"], a), "secs": round(time.time() - t, 1)})
        print(f"[{label}] {i:2}/{len(EVAL['questions'])} {rows[-1]['score']} {item['site']} | expected: {item['expected']}")
        print(f"     ALLaM: {a[:160]}\n")
    return rows

def summary(rows, label):
    graded = [r for r in rows if r["score"] != "👀"]
    right = sum(r["score"] == "✅" for r in graded)
    print(f"{label}: numeric facts correct {right}/{len(graded)} ({100 * right / max(1, len(graded)):.0f}%) · by eye: {len(rows) - len(graded)}")
    return right, len(graded)

baseline = run(lambda item: item["q"], "alone")

In [ ]:
base_score = summary(baseline, "ALLaM alone")

## Grounded run — ALLaM + Athr's verified data
Same 24 questions, but each one comes with that site's verified text from Athr (facts + story).
This is how Athr would really use a language model: it explains *our* facts instead of recalling from memory.

In [ ]:
GROUNDED = """أنت مرشد سياحي لتطبيق «أثر» في منطقة حائل.
أجب عن السؤال اعتمادًا على المعلومات الموثّقة التالية فقط. إن لم تجد الإجابة فيها فقل: «لا تتوفر لديّ معلومة موثّقة».

المعلومات الموثّقة:
{context}

السؤال: {q}"""

grounded = run(lambda item: GROUNDED.format(context=CONTEXT[item["site_id"]], q=item["q"]), "grounded")

In [ ]:
g_score = summary(grounded, "ALLaM + Athr data")
print()
print(f"Alone:    {base_score[0]}/{base_score[1]}")
print(f"Grounded: {g_score[0]}/{g_score[1]}")
print("\nSide by side (non-numeric answers — judge by eye):")
for b, g in zip(baseline, grounded):
    if b["score"] == "👀":
        print(f"\n• {b['site']} — expected: {b['expected']}")
        print(f"  alone:    {b['answer'][:140]}")
        print(f"  grounded: {g['answer'][:140]}")

## Open questions — would ALLaM make a good tour guide?

In [ ]:
OPEN = [
    "أنا سائح أزور حائل لأول مرة. اشرح لي نقوش جبة الصخرية في خمس جمل بأسلوب مرشد سياحي.",
    "ما الفرق بين النقوش الثمودية والنبطية؟ وأين أجدها في منطقة حائل؟",
    "خطط لي يومًا واحدًا في حائل يجمع التاريخ والطبيعة، مع أفضل وقت لكل محطة.",
]
open_answers = []
for q in OPEN:
    a = ask(q, max_new_tokens=500)
    open_answers.append({"q": q, "answer": a})
    print("Q:", q)
    print(a)
    print("-" * 60)

# the first question again, grounded — the real Athr experience
jubbah = [s for s in EVAL["sites"] if s["id"] == "jubbah"][0]
a = ask(GROUNDED.format(context=jubbah["context"], q=OPEN[0]), max_new_tokens=500)
open_answers.append({"q": OPEN[0] + "  [grounded]", "answer": a})
print("Q (grounded):", OPEN[0])
print(a)

In [ ]:
import os
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."   # Kaggle or Colab
with open(f"{OUT}/allam_athr_results.json", "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "alone": baseline, "grounded": grounded, "open": open_answers,
               "score_alone": base_score, "score_grounded": g_score}, f, ensure_ascii=False, indent=1)
lines = [f"**Alone:** {base_score[0]}/{base_score[1]} · **Grounded:** {g_score[0]}/{g_score[1]}", "",
         "| Site | Expected | Alone | Grounded |", "|---|---|---|---|"]
for b, g in zip(baseline, grounded):
    cut = lambda r: r["score"] + " " + r["answer"][:90].replace("|", "/").replace("\n", " ")
    lines.append(f"| {b['site']} | {b['expected']} | {cut(b)} | {cut(g)} |")
with open(f"{OUT}/allam_athr_results.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print(f"Saved -> {OUT}/allam_athr_results.json + .md")